In [ ]:
# =============================================================================
# Long and Short Pause Statistics Computation
#
# PURPOSE:
# This notebook computes detailed statistics for long and short pause events
# from processed and relabelled speech/pause data.
#
# INPUTS:
# - *_50_150_final.csv files (output from ATAS_event_relabelling notebook)
#
# OUTPUTS:
# - Summary statistics for:
#     - Long pauses
#     - Short pauses
#     - Duration distributions
#     - Event counts
#
# PROCESS:
# 1. Load final event-level CSV files
# 2. Identify long vs short pauses based on duration thresholds
# 3. Compute descriptive statistics (mean, std, counts, etc.)
#
# NOTE:
# This notebook should be run AFTER:
# 1. Automated_ATAS_Streamlined_CWS_CWNS.ipynb
# 2. ATAS_event_relabelling.ipynb
# =============================================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statistics
from scipy.stats import variation
from scipy.stats import mannwhitneyu

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

In [ ]:
# Load ATAS multiple file analysis output files (Outputs from - Streamlined_ATAS_CWS.ipynb/ Streamlined_ATAS_CWNS.ipynb)
df_1 = pd.read_csv(base_dir + '/Stat_csv_files/CWNS_All_files_together_50_ms_1_win_50_150_csv.csv') #Change path as required #CWNS
df_2 = pd.read_csv(base_dir + '/Stat_csv_files/CWS_All_files_together_50_ms_1_win_50_150_csv.csv') #Change path as required #CWS

details_files_with_ssi = pd.read_csv(base_dir + '/Stat_csv_files/CWNS_CWS_all_demo_and_score_details.csv')

# CSV files with Participant details
CWNS_par = pd.read_csv(base_dir + '/Stat_csv_files/CWNS_details.csv') #Change path as required
CWS_par = pd.read_csv(base_dir + '/Stat_csv_files/CWS_details.csv') #Change path as required

individual_csv_files_path =  base_dir + '/Individual_OutputCSV_Files/' #Change path as required

# Rows where Group == 'CWS', select only required columns
ssi_scores_and_other_details_cws = details_files_with_ssi.loc[
    details_files_with_ssi['Group'] == 'CWS',
    ['ID', 'SSI', '%SS','Age','Sex']
]

# Rows where Group == 'CWNS', select only required columns
ssi_scores_and_other_details_cwns = details_files_with_ssi.loc[
    details_files_with_ssi['Group'] == 'CWNS',
    ['ID', 'SSI', '%SS','Age','Sex']
]

In [ ]:
%run -i "$base_dir/Scripts/Long_short_pause_compute_functions.ipynb"  #Change path as required

In [ ]:
columns_to_remove = ['Long Pauses', 'Short Pauses'] # if present
df_1_cleaned = df_1.drop(columns=columns_to_remove, errors='ignore')
df_2_cleaned = df_2.drop(columns=columns_to_remove, errors='ignore')
df_all = pd.concat([df_1_cleaned, df_2_cleaned], axis=0, ignore_index=True)

In [ ]:
# Merge participant details
df_all_1 = merge_dataframes_on_filename(df_all, CWNS_par, CWS_par)
print(df_all_1.columns)
# Calculate long and short pause metrics

# Define pause duration thresholds used to classify pauses as short or long

pause_threshold = 0.15 # the threshold for the long and short pause categorization in sec 
# pause event >= pause_threshold - long pause 
# pause event < pause_threshold - short pause 

for i, row in df_all_1.iterrows():
    filename = row['ID']
    print(filename)
    csv_filename_1 = filename.split('.wav')[0] + '_50_150_final.csv'
    csv_filename = individual_csv_files_path + csv_filename_1
    process_pause_durations(csv_filename, df_all_1, i, pause_threshold)

# Calculate the speech rate
calculate_speech_rate(df_all_1)


In [ ]:
# Remove if not applicable

# Merge SSI scores
df_all_2 = merge_ssi_scores(df_all_1, ssi_scores_and_other_details_cws,ssi_scores_and_other_details_cwns) # Remove if not applicable

In [ ]:
# Export final csv
output_csv_path = base_dir + '/Stat_csv_files/CWNS_CWS_all_details.csv' # Specify your output path
df_all_2.to_csv(output_csv_path, index=False)  # Save the DataFrame to a CSV file

In [ ]:
# Set display options to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # Prevent wrapping of columns
pd.set_option('display.max_rows', None)  # Optional: show all rows if needed